 **The database have two jobs: manage the async SQLAlchemy engine and create the `agent_memory` table.**
 ---
 ### Engine setup
 `create_engine` with a few important params:
 - `pool_size=10, max_overflow=20` — up to 30 concurrent DB connections before it starts waiting
 - `pool_pre_ping=True` — tests connections before using them, avoids "connection closed" errors after idle periods
 - `pool_recycle=3600` — forces connection renewal every hour, prevents stale connections on long-running servers
 - `echo=settings.debug` — SQL query logging only in debug mode
 `expire_on_commit=False` on the sessionmaker — without this, accessing model attributes after a commit triggers a lazy reload, which breaks in async context.
 ---
 ![agent_memory](image/db_1.png)

In [ ]:
from typing import AsyncGenerator, Optional
import asyncio

from sqlalchemy import text
from sqlalchemy.ext.asyncio import AsyncEngine, AsyncSession, create_async_engine
from sqlalchemy.orm import sessionmaker
from app.core.config import get_settings

_engine: Optional[AsyncEngine] = None
_sessionmaker: Optional[sessionmaker] = None
_init_lock: Optional[asyncio.Lock] = None


def _get_init_lock() -> asyncio.Lock:
    global _init_lock
    if _init_lock is None:
        _init_lock = asyncio.Lock()
    return _init_lock


def _create_engine() -> AsyncEngine:
    global _engine, _sessionmaker
    if _engine is None:
        settings = get_settings()
        if not settings.database_url:
            raise RuntimeError("DATABASE_URL is required to create the async engine")
        _engine = create_async_engine(
            str(settings.database_url),
            echo=settings.debug,
            future=True,
            pool_size=10,
            max_overflow=20,
            pool_pre_ping=True,
            pool_recycle=3600,
        )
        _sessionmaker = sessionmaker(
            bind=_engine,
            class_=AsyncSession,        # BUG 1 fix
            expire_on_commit=False,
            autoflush=False,
        )
    return _engine

 ##  Bugs fixing
 **Bug 1 — sessionmaker missing `class_=AsyncSession`.**
 Default sessionmaker creates sync sessions. Calling `await session.execute()` on a sync session raises a confusing error. Explicit `class_=AsyncSession` fixes it.
 we can illustrate like this


 ![The Double-Checked Locking](image/db_3.png)

In [ ]:
def get_engine() -> AsyncEngine:
    return _create_engine()              # BUG 2 fix


def get_async_session() -> AsyncSession:
    if _sessionmaker is None:
        _create_engine()
    return _sessionmaker()

 **Bug 2 — `get_engine()` called before `_create_engine()`.**

 If something calls `get_engine()` before `init_database()` runs, `_engine` is still `None`. Routing through `_create_engine()` instead of returning `_engine` directly handles lazy initialization safely.

 ---
 ### `get_db`

 FastAPI dependency — yields a session, cleans up automatically. So we  use it like this :
 ```python
 async def my_route(db: AsyncSession = Depends(get_db)):
     ...
 ```


 ---

 ### `close_db`
 Called in the FastAPI lifespan on shutdown. `engine.dispose()` closes all pooled connections cleanly. Sets both globals back to `None` so the app can reinitialize if needed.

In [ ]:
async def get_db() -> AsyncGenerator[AsyncSession, None]:
    if _sessionmaker is None:
        _create_engine()
    async with _sessionmaker() as session:
        yield session


async def init_database() -> None:      # BUG 3 fix : one definition with lock
    async with _get_init_lock():
        if _engine is not None:
            return
        _create_engine()

    async with get_engine().begin() as conn:
        await conn.execute(text("SELECT 1"))
        await conn.execute(text("""
            CREATE TABLE IF NOT EXISTS agent_memory (
                user_id     TEXT PRIMARY KEY,
                memory_data JSONB NOT NULL,
                updated_at  TIMESTAMP WITH TIME ZONE NOT NULL DEFAULT NOW()
            )
        """))


async def close_db() -> None:
    global _engine, _sessionmaker
    if _engine is not None:
        await _engine.dispose()
        _engine = None
        _sessionmaker = None

 # So let's look the final architecture
# FastAPI asks -> Session holds it -> Engine drives the truck -> Pool keeps the truck running smoothly -> and it all dumps right into that simple JSON table.


 ![Architecture](image/db_2.png)